<a href="https://colab.research.google.com/github/xidoudou/ai-agents/blob/main/rag02_courage_to_be_disliked.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install & Import**

In [ ]:
!pip install -q sentence_transformers openai pypdf

import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from pypdf import PdfReader
from google.colab import userdata, drive

drive.mount("/content/drive")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
client = OpenAI(api_key = userdata.get("OPENAI_API_KEY"))


**Config**

In [ ]:
pdf_path = userdata.get("RAG02_PDF_PATH")
CHUNK_SIZE = 200
OVERLAP = 50
TOP_K = 4

**Load**

In [ ]:
reader = PdfReader(pdf_path)
pages = []

for i, page in enumerate(reader.pages):
  text = page.extract_text() or ""
  pages.append({"page": i + 1, "text": text})

print(f"{len(pages)} pages in total")


**Quality Check**

In [ ]:
for p in [pages[10], pages[20], pages[200]]:
  print(f"======== Page {p['page']} ========")
  print(p['text'][:500])
  print()

**Chunking**

In [ ]:
def chunk_pages(pages, chunk_size = CHUNK_SIZE, overlap = OVERLAP):
  chunks = []
  step = chunk_size - overlap

  for p in pages:
    words = p['text'].split()

    for i in range(0, len(words), step):
      chunk = " ".join(words[i: i + chunk_size])
      chunks.append({"id": len(chunks), "page": p['page'], "text": chunk})

  return chunks

chunks = chunk_pages(pages)
print(f"{len(chunks)} chunks in total")
print(chunks[100])
print(chunks[101])

**Ingestion**

In [ ]:
texts = []
for c in chunks:
  texts.append(c['text'])
embeddings = embed_model.encode(texts, show_progress_bar = True)
print(embeddings.shape)

**Retrieval**

In [ ]:
def retrieve(query, k = TOP_K):
  query_embedding = embed_model.encode(query)
  scores = embed_model.similarity(query_embedding, embeddings)[0]
  top_indices = np.argsort(scores.numpy())[::-1][:k]
  return [(chunks[i], float(scores[i])) for i in top_indices]

In [ ]:
for chunk, score in retrieve("What is the separation of tasks?"):
    print(f"page {chunk['page']}  score={score:.3f}  {chunk['text'][:80]}...")

**Generation**

In [ ]:
def generate_rag_response(query, k = TOP_K):
  results = retrieve(query, k)
  context = "\n".join(f"[Page {c['page']}] {c['text']}" for c, score in results)


  prompt = f"""
  You are an assistant that answers questions about the book "The Courage to Be Disliked". Please answer users' Questions according to the following rules:
  1. Only answer the content which are included in following Context, don't generate any answer using your own knowledge.
  2. If you don't find any answer of the question from Context, please reply "I don't find any information to this query"
  3. At the end of the answer, cite the page numbers you used, e.g., (Source: page 57)

  Context:
  {context}

  Question: {query}
  """

  response = client.responses.create(
      model = "gpt-6-astra",
      input = prompt
  )
  return response.output_text, results

**Test**

In [ ]:
answer, results = generate_rag_response("What is teleology?")
print(answer)
print(f"results: {results}")

In [ ]:
answer, results = generate_rag_response("What is teleology?")

print("Answer:\n", answer)
print("\nRetrieved:")
for c, score in results:
    print(f"  page {c['page']}  score={score:.3f}  {c['text'][:800]}...")